# 01 Primer exercise: one holding, end to end
Find one holding in the raw schedule of investments, see how the SEC's flat file stores it, compute its price, then find the same holding in the filing itself.

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "params.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from lookthrough import paths, acquire, load, mapping, holdings, clean, reconcile, report, tables, pipeline
P = paths.load_params(ROOT)
RDIR = paths.release_dir(P, ROOT)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)
print("release folder:", RDIR.relative_to(ROOT), "| fixture mode:", P["data"]["use_fixture"])

In [ ]:
header = load.soi_header(RDIR)
print(f"soi.tsv has {len(header)} columns; the first 13:")
print(header[:13])

In [ ]:
sub, tag = load.load_sub(RDIR), load.load_tag(RDIR)
header = load.soi_header(RDIR)
colmap = mapping.map_soi_columns(header, tag)
soi = load.load_soi(RDIR, usecols=mapping.soi_usecols(header, colmap))   # only the columns we use
soi = soi[soi["adsh"].isin(sub["adsh"])]                                  # drop superseded filings
canon = holdings.split_periods(mapping.to_canonical(soi, colmap), sub)  # current period from SUB
FV = mapping.PRESET["fair_value"]
cur = canon[canon["is_current"] & canon["identifier"].notna()]
filing = cur.groupby("adsh").size().idxmax()            # the filing with the most rows
ident = cur[cur["adsh"] == filing]["identifier"].value_counts().index[0]   # an identifier with several rows
rows = canon[(canon["adsh"] == filing) & (canon["identifier"] == ident)]
print("Filing:", filing, "|", rows["name"].iloc[0])
print("Identifier:", ident)
print(f"In this filing's current schedule, this holding occupies {len(rows)} SOI row(s), one per group of facts:")
print(rows[["soi_row", "ddate", "fair_value", "cost", "principal", "interest_rate", "pct_net_assets"]].to_string(index=False))

In [ ]:
merged = holdings.coalesce_holdings(rows).iloc[0]
price = 100 * merged["fair_value"] / merged["principal"] if merged["principal"] else float("nan")
print(f"Merged: fair value {merged['fair_value']:,.0f}, principal {merged['principal']:,.0f} -> price {price:.1f}")
borrower = ident.split(" | ")[1] if " | " in ident else ident[:30]
prior = canon[(canon["adsh"] == filing) & ~canon["is_current"] & canon["identifier"].fillna("").str.contains(borrower, regex=False)]
if len(prior):
    print("The prior fiscal year-end schedule names it differently (the coupon in the identifier changed):")
    print("  ", prior["identifier"].iloc[0])
print("Open the filing and find this holding in its schedule of investments:")
print(rows["inlineurl"].iloc[0])

**Your turn.** Open the link (real releases only; the fixture's links are placeholders). Find the holding in the human-readable schedule and compare every field: principal, cost, fair value, rate, spread, maturity. Note anything the flat file split, dropped or changed.